# Part 1 — Supervised Fine-Tuning and DARE

**1.1** LoRA instruction tuning of `Qwen/Qwen2.5-1.5B-Instruct` on the first 60% of `medalpaca/medical_meadow_medqa`.

**1.2** DARE (Drop And REscale) applied to the SFT delta, merged with `mergekit`, with a drop-rate sweep selected on the validation split.

In [ ]:
# --- environment -----------------------------------------------------------
import os, sys, glob, shutil, zipfile
from pathlib import Path

os.environ["HF_HOME"] = "/kaggle/temp/hf"                  # dataset + model cache
os.environ["SAFEALIGN_ROOT"] = "/kaggle/temp/safealign"    # big artifacts, off the 20 GB quota
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

try:                                                        # Add-ons -> Secrets -> HF_TOKEN
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("HF_TOKEN secret not found:", e)

!pip install -q -U "transformers>=4.44" "peft>=0.12" "datasets>=2.20" "accelerate>=0.33" \
    rouge-score sacrebleu nltk mergekit

# --- locate the project inside the attached Kaggle Dataset -----------------
REPO = "/kaggle/working/safety-alignment-llm"

def find_source():
    # Kaggle auto-extracts uploaded archives, so the dataset may hold either
    # the unpacked folder or the original .zip. Handle both.
    hits = glob.glob("/kaggle/input/**/src/safealign/config.py", recursive=True)
    if hits:
        return ("dir", str(Path(hits[0]).parents[2]))
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    if zips:
        return ("zip", zips[0])
    raise FileNotFoundError("Attach the dataset holding the project (Add Input -> Datasets)")

if not os.path.exists(REPO):
    kind, src = find_source()
    if kind == "zip":
        with zipfile.ZipFile(src) as z:
            z.extractall("/kaggle/working")
    else:
        shutil.copytree(src, REPO)                          # /kaggle/input is read-only
    print("project from", kind, src)

sys.path.insert(0, f"{REPO}/src")

import torch
print(torch.__version__, "| GPUs:", torch.cuda.device_count(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
from safealign.config import CFG
CFG.paths.ensure(); print("artifacts ->", CFG.paths.artifacts)


## 1.1 Instruction tuning

Hyper-parameters are declared in `safealign/config.py` and written to `train_stats.json` next to the adapter, so the report can quote them without transcription errors.

In [ ]:
from safealign.config import CFG, MODEL_SFT
from safealign.sft import train_medical_sft
from safealign.model_utils import merge_lora

adapter = train_medical_sft()
merged = merge_lora(adapter, CFG.paths.artifacts / f'{MODEL_SFT}_merged')
print(open(adapter / 'train_stats.json').read())

## 1.2 DARE

`density = 1 - p`. mergekit's `dare_linear` performs the Bernoulli drop and the `1/(1-p)` rescale; the reference implementation in `safealign/dare.py` is used to verify the arithmetic.

In [ ]:
from safealign.dare import delta_stats, sweep_drop_rates

print(delta_stats(str(merged)))
sweep = sweep_drop_rates(merged, n_val=150)
sweep['best_p'], sweep['sweep']

In [ ]:
import pandas as pd
pd.DataFrame(sweep['sweep']).T.rename_axis('drop rate p')